# 9 — Receiver-gated, drone-balanced statistics (Phase 6, post hoc)

OpenSpec change `receiver-gated-adaptation`. **Post-hoc follow-up, disclosed:** designed after the frozen Phase 5
target results (notebook 7) were seen, and reported beside them, never in their place. A 2026-10-26 spike ran five
balanced-buffer settings on the dev cells only.

Same streams as Phase 5 (same recordings, order, centring and active windows), plus the stream estimator's
noise-floor drift as the receiver gate's input. Methods: `balanced`, `gated_continuous`, `gated_balanced`, and an
oracle diagnostic (`gated_continuous` with the gate forced ON exactly at the change point).

In [1]:
import os
import json
import itertools
import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from sklearn.metrics import f1_score
from tqdm import tqdm

from rfml_uav.adapt.stream import energy, run_stream
from rfml_uav.eval.stats import paired_test
from rfml_uav.data.normalise import session_normalise, stream_normalise
from rfml_uav.drone_detect.dataset import DRONES
from rfml_uav.eval.splits import make_splits, undersample
from rfml_uav.nn.models import get_model

load_dotenv()
DATA = os.getenv("DATA_PATH")
P1, P2, P3, P4 = (os.path.join(DATA, f"results/paper3/phase{i}") for i in (1, 2, 3, 4))
P5, P6 = (os.path.join(DATA, f"results/paper3/phase{i}") for i in (5, 6))
os.makedirs(os.path.join(P6, "streams"), exist_ok=True)
CKPT = os.path.join(DATA, "checkpoints/paper3/noisenorm")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = True, False

CLASSES = list(DRONES)
SEEDS, K_FOLDS = [0, 1, 2, 3, 4], 5
BETA, K_WARM = 0.01, 50  # Phase 4 stream estimator
SHIFT_A = ["BLUE", "WIFI", "BOTH"]
TARGETS = SHIFT_A + ["awgn_2", "awgn_3", "gain_3", "agc_3", "adc_3", "tilt_4"]
DEV = ["awgn_1", "gain_2", "adc_2", "tilt_3"]
# notebook 5's ids: the stream order and the E3 splits share the rng key [seed, fold, TARGET_ID]
TARGET_ID = {t: i for i, t in enumerate(SHIFT_A + ["awgn_2", "awgn_3", "gain_3", "adc_3", "tilt_4"] + DEV + ["agc_3"])}

In [2]:
F = dict(np.load(os.path.join(P1, "features.npz")))
F["y"] = np.array([CLASSES.index(d) for d in F["drone"]])
clean, active = F["subset"] == "CLEAN", F["active"]
REC = {r: rows[np.argsort(F["start"][rows], kind="stable")]  # rec -> all rows, active and noise interleaved by time
       for r, rows in pd.Series(np.arange(len(F["y"]))).groupby(F["rec_id"]).apply(np.array).items()}
XS = session_normalise(F["X"], F["rec_id"], ~active)  # session estimator: calibration rows only

CLEAN_ROWS = np.flatnonzero(clean)
STEPPED = np.array([np.random.default_rng([0, 99, r]).random() < 0.05 for r in CLEAN_ROWS])  # notebooks 5/6 mask
_imp = {}


def impaired(cell):
    """Raw impaired features of every CLEAN row (active and noise) of a shift-B cell, row-aligned with CLEAN_ROWS."""
    if cell not in _imp:
        base = "gain_3" if cell == "agc_3" else cell
        X = F["X"][CLEAN_ROWS].copy() if cell == "agc_3" else np.zeros((len(CLEAN_ROWS), F["X"].shape[1]), np.float32)
        n = 0
        for d in ("dronedetect", "dronedetect_noise"):
            g = np.load(os.path.join(P2, d, base + ".npz"))
            pos = np.searchsorted(CLEAN_ROWS, g["row"])
            assert np.array_equal(CLEAN_ROWS[pos], g["row"])
            keep = STEPPED[pos] if cell == "agc_3" else np.ones(len(pos), bool)
            X[pos[keep]] = g["X"][keep]
            n += len(pos)
        assert cell == "agc_3" or n == len(CLEAN_ROWS)
        _imp[cell] = X
    return _imp[cell]


def raw(target, rows):
    if target == "clean" or target in SHIFT_A:
        return F["X"][rows]
    return impaired(target)[np.searchsorted(CLEAN_ROWS, rows)]


MODELS = {}
for seed, fold in itertools.product(SEEDS, range(K_FOLDS)):
    m = get_model("CNN", input_dim=256, n_classes=len(CLASSES), linear_tail=True)
    m.load_state_dict(torch.load(os.path.join(CKPT, f"active_linear_s{seed}_f{fold}.pt"), map_location=device))
    MODELS[(seed, fold)] = m.to(device).eval()


def source_split(seed, fold):
    """(train, val, test) rows of the centred source model, rebuilt as in Phase 4."""
    idx = undersample(np.flatnonzero(clean & active), F["y"], seed)
    tr, va, te = make_splits(F["y"][idx], F["rec_id"][idx], "recording", n_folds=K_FOLDS, seed=seed)[fold]
    run = np.load(os.path.join(P4, f"runs/active_linear_s{seed}_f{fold}.npz"))
    assert np.array_equal(idx[te], run["test_idx"])
    return idx[tr], idx[va], idx[te]

In [3]:
def build_stream(target, seed, fold):
    """Phase 5 receiver-condition stream (Phase 5 design D7) -> dict(X, active, y, rows, change_point, drift).

    X / active cover every window (noise windows are only for centring); y / rows are per active window, aligned
    with run_stream's outputs; change_point = index of the first target window among the active windows.
    """
    _, _, te = source_split(seed, fold)
    test_rows = np.load(os.path.join(P3, "splits", f"{target}_s{seed}_f{fold}.npz"))["test_rows"]
    rng = np.random.default_rng([seed, fold, TARGET_ID[target]])
    prefix, tgt = (np.array(sorted(set(F["rec_id"][r]))) for r in (te, test_rows))
    prefix, tgt = prefix[rng.permutation(len(prefix))], tgt[rng.permutation(len(tgt))]
    rp, rt = np.concatenate([REC[r] for r in prefix]), np.concatenate([REC[r] for r in tgt])
    rows = np.r_[rp, rt]
    Xn, valid, drift = stream_normalise(np.r_[raw("clean", rp), raw(target, rt)], ~active[rows], beta=BETA, k=K_WARM,
                                        return_drift=True)
    act = active[rows] & valid
    return {"X": Xn, "active": act, "y": F["y"][rows][act], "rows": rows[act], "change_point": int(act[:len(rp)].sum()),
            "drift": drift,
            "prefix_recs": prefix, "target_recs": tgt}


def check_stream(s, target, seed, fold):
    tr, va, _ = source_split(seed, fold)
    recs = F["rec_id"][s["rows"]]
    assert not set(recs) & set(F["rec_id"][np.r_[tr, va]]), "stream recording seen in training"
    cp = s["change_point"]
    assert set(recs[:cp]) == set(s["prefix_recs"]) and set(recs[cp:]) == set(s["target_recs"])
    assert (F["subset"][s["rows"][:cp]] == "CLEAN").all()
    assert recs[cp] == s["target_recs"][0]  # shift B may replay the last prefix recording first
    for seg in (slice(0, cp), slice(cp, None)):  # shift B replays prefix recordings, impaired, in the target segment
        for r in np.unique(recs[seg]):
            assert np.all(np.diff(F["start"][s["rows"][seg][recs[seg] == r]]) > 0), r

In [4]:
def calib_energy(seed, fold):
    """Source energies on its own CLEAN validation rows (session-centred). No labels used."""
    _, va, _ = source_split(seed, fold)
    with torch.no_grad():
        X = torch.as_tensor(XS[va], device=device).unsqueeze(1)
        return torch.cat([energy(MODELS[(seed, fold)](X[i:i + 4096])) for i in range(0, len(X), 4096)]).cpu().numpy()


def run(s, seed, fold, method, hp, calib, drift=None):
    return run_stream(MODELS[(seed, fold)], torch.as_tensor(s["X"], device=device).unsqueeze(1), s["active"], method,
                      hp, calib, seed=1000 * seed + fold, drift=s["drift"] if drift is None else drift)


def score(s, o):
    cp, y = s["change_point"], s["y"]
    f = lambda a, b: f1_score(a, b, average="macro", labels=range(len(CLASSES)))
    return {"vote_f1": f(y[cp:], o["vote"][cp:]), "window_f1": f(y[cp:], o["pred"][cp:]),
            "prefix_vote_f1": f(y[:cp], o["vote"][:cp]), "rejection": 1 - o["known"].mean(),
            "events": o["cost"]["events"], "bwd_per_window": o["cost"]["bwd_per_window"],
            "fwd_per_window": o["cost"]["fwd_per_window"]}




with open(os.path.join(P5, "stream_hparams.json")) as fh:  # frozen Phase 5 settings: W, q, M, R, C, ...
    P5_HPS = json.load(fh)["methods"]
CALIB = {(seed, fold): calib_energy(seed, fold) for seed, fold in MODELS}


def p5(target, seed, fold):
    return np.load(os.path.join(P5, "streams", f"{target}_s{seed}_f{fold}.npz"))


def check_same_as_phase5(s, target, seed, fold):
    r = p5(target, seed, fold)
    assert np.array_equal(s["rows"], r["rows"]) and s["change_point"] == int(r["change_point"])


for target, seed, fold in [("BOTH", 0, 0), ("agc_3", 1, 3), ("tilt_4", 4, 2)]:
    s = build_stream(target, seed, fold)
    check_stream(s, target, seed, fold)
    check_same_as_phase5(s, target, seed, fold)
    assert np.array_equal(run(s, seed, fold, "source", P5_HPS["source"], CALIB[(seed, fold)])["vote"],
                          p5(target, seed, fold)["source_vote"])
print("streams identical to Phase 5; source-only reproduces the Phase 5 votes")

streams identical to Phase 5; source-only reproduces the Phase 5 votes


## Harm-aware dev tuning (design D7)

Seed-0 models on the Phase 5 dev cells, 5 folds each. Objective J = mean target vote F1 − mean max(0, source prefix
vote F1 − method prefix vote F1). Within 0.1 pp of the best J → fewer forward passes per window. W, q, M, R, C stay
as in `phase5/stream_hparams.json`.

In [5]:
DEV_STREAMS = {(c, f): build_stream(c, 0, f) for c, f in itertools.product(DEV, range(K_FOLDS))}
SRC_PREFIX = {k: score(s, run(s, 0, k[1], "source", P5_HPS["source"], CALIB[(0, k[1])]))["prefix_vote_f1"]
              for k, s in DEV_STREAMS.items()}
tuning_path = os.path.join(P6, "dev_tuning.csv")
done = pd.read_csv(tuning_path) if os.path.exists(tuning_path) else pd.DataFrame(columns=["stage", "method", "hp"])
log = [done]


def evaluate(stage, method, hp):
    key = json.dumps(hp, sort_keys=True)
    old = done[(done.stage == stage) & (done.method == method) & (done.hp == key)]
    if len(old):
        return old
    rows = []
    for (c, f), s in DEV_STREAMS.items():
        o = run(s, 0, f, method, hp, CALIB[(0, f)])
        sc = score(s, o)
        rows.append({"stage": stage, "method": method, "hp": key, "cell": c, "fold": f, **sc,
                     "harm": max(0.0, SRC_PREFIX[(c, f)] - sc["prefix_vote_f1"]), "updates": o["cost"]["updates"],
                     "on_fraction": o["cost"]["on_fraction"]})
    df = pd.DataFrame(rows)
    log.append(df)
    pd.concat(log).to_csv(tuning_path, index=False)
    return df


def pick(stage, method, grid):
    res = pd.DataFrame([{"hp": json.dumps(hp, sort_keys=True), **evaluate(stage, method, hp)
                        .drop(columns=["stage", "method", "hp", "cell", "fold"]).mean().to_dict()} for hp in grid])
    res["J"] = res.vote_f1 - res.harm
    best = res[res.J >= res.J.max() - 0.001].sort_values("fwd_per_window").iloc[0]
    return json.loads(best.hp), res


base = dict(P5_HPS["source"])
w1, s1 = pick(1, "balanced", [{**base, "Bc": Bc, "D": D, "m": m}
                              for Bc, D, m in itertools.product((200, 500), (3, 5), (0.01, 0.1))])
w2, s2 = pick(2, "gated_continuous", [{**base, "phi_on": phi, "m": m}
                                      for phi, m in itertools.product((0.3, 0.5, 0.7), (0.01, 0.1))])
w3, s3 = pick(3, "gated_balanced", [{**w1, "phi_on": phi} for phi in (0.3, 0.5, 0.7)])
HP6 = {"balanced": w1, "gated_continuous": w2, "gated_balanced": w3,
       "oracle": {**w2, "phi_on": 0.5}}  # oracle: drift forced 0 before / 1 after the change point
WINNERS = {"1": {k: w1[k] for k in ("Bc", "D", "m")}, "2": {k: w2[k] for k in ("phi_on", "m")},
           "3": {"phi_on": w3["phi_on"]}}

hp_path = os.path.join(P6, "hparams.json")
assert not any(f.endswith(".npz") for f in os.listdir(os.path.join(P6, "streams"))) or os.path.exists(hp_path), \
    "target streams exist without a saved hyperparameter file"
if not os.path.exists(hp_path):  # frozen once, before any target stream
    with open(hp_path, "w") as fh:
        json.dump({"objective": "vote_f1 - max(0, source prefix vote F1 - prefix vote F1)", "stage_winners": WINNERS,
                   "methods": HP6}, fh, indent=1)
print(json.dumps(WINNERS))
pd.set_option("display.width", 250)
pd.concat([s1.assign(stage=1, method="balanced"), s2.assign(stage=2, method="gated_continuous"),
           s3.assign(stage=3, method="gated_balanced")])[
    ["stage", "method", "hp", "J", "vote_f1", "harm", "prefix_vote_f1", "fwd_per_window", "updates", "on_fraction"]].round(4)

{"1": {"Bc": 200, "D": 5, "m": 0.01}, "2": {"phi_on": 0.7, "m": 0.1}, "3": {"phi_on": 0.7}}


,stage,method,hp,J,vote_f1,harm,prefix_vote_f1,fwd_per_window,updates,on_fraction
0,1,balanced,"{""B"": 1000, ""Bc"": 200, ""D"": 3, ""W"": 25, ""lam"":...",0.6017,0.6056,0.0039,0.8590,5.7614,115.15,1.0000
1,1,balanced,"{""B"": 1000, ""Bc"": 200, ""D"": 3, ""W"": 25, ""lam"":...",0.5954,0.6237,0.0282,0.8310,5.8054,115.15,1.0000
2,1,balanced,"{""B"": 1000, ""Bc"": 200, ""D"": 5, ""W"": 25, ""lam"":...",0.6048,0.6062,0.0015,0.8606,5.4964,103.65,1.0000
3,1,balanced,"{""B"": 1000, ""Bc"": 200, ""D"": 5, ""W"": 25, ""lam"":...",0.6011,0.6216,0.0205,0.8399,5.5063,103.65,1.0000
4,1,balanced,"{""B"": 1000, ""Bc"": 500, ""D"": 3, ""W"": 25, ""lam"":...",0.5980,0.6024,0.0044,0.8580,12.6196,113.95,1.0000
5,1,balanced,"{""B"": 1000, ""Bc"": 500, ""D"": 3, ""W"": 25, ""lam"":...",0.5814,0.6095,0.0281,0.8310,12.7273,113.95,1.0000
6,1,balanced,"{""B"": 1000, ""Bc"": 500, ""D"": 5, ""W"": 25, ""lam"":...",0.6027,0.6041,0.0014,0.8604,11.8478,100.45,1.0000
7,1,balanced,"{""B"": 1000, ""Bc"": 500, ""D"": 5, ""W"": 25, ""lam"":...",0.5924,0.6102,0.0178,0.8428,11.8828,100.45,1.0000
0,2,gated_continuous,"{""B"": 1000, ""D"": 3, ""W"": 25, ""lam"": 0.3, ""m"": ...",0.6055,0.6063,0.0008,0.8588,1.3554,52.70,0.4248
1,2,gated_continuous,"{""B"": 1000, ""D"": 3, ""W"": 25, ""lam"": 0.3, ""m"": ...",0.6390,0.6565,0.0175,0.8432,1.3573,52.70,0.4248


## Targets (run once)

In [6]:
with open(os.path.join(P6, "hparams.json")) as fh:  # the only source of Phase 6 hyperparameters from here on
    HP6 = json.load(fh)["methods"]
P6_METHODS = ["balanced", "gated_continuous", "gated_balanced", "oracle"]


def stream_path(target, seed, fold):
    return os.path.join(P6, "streams", f"{target}_s{seed}_f{fold}.npz")


def oracle_drift(s):
    d = np.zeros(len(s["active"]))
    d[np.flatnonzero(s["active"])[s["change_point"]]:] = 1.0  # protocol's change point: diagnostic only
    return d


def run6(s, seed, fold, method):
    if method == "oracle":
        return run(s, seed, fold, "gated_continuous", HP6["oracle"], CALIB[(seed, fold)], drift=oracle_drift(s))
    return run(s, seed, fold, method, HP6[method], CALIB[(seed, fold)])


def run_target(target, seed, fold):
    path = stream_path(target, seed, fold)
    if os.path.exists(path):
        return
    s = build_stream(target, seed, fold)
    check_stream(s, target, seed, fold)
    check_same_as_phase5(s, target, seed, fold)
    res = {"rows": s["rows"], "change_point": s["change_point"],
           "drift": s["drift"][s["active"]].astype(np.float16)}
    for method in P6_METHODS:
        o = run6(s, seed, fold, method)
        res.update({f"{method}_pred": o["pred"].astype(np.int8), f"{method}_vote": o["vote"].astype(np.int8),
                    f"{method}_known": o["known"], f"{method}_on": np.array(o["cost"].pop("on"), np.int8),
                    f"{method}_cost": json.dumps(o["cost"])})
    np.savez(path + ".tmp.npz", **res)
    os.replace(path + ".tmp.npz", path)


for target, seed, fold in tqdm(list(itertools.product(TARGETS, SEEDS, range(K_FOLDS)))):
    run_target(target, seed, fold)
files = os.listdir(os.path.join(P6, "streams"))
assert len(files) == len(TARGETS) * len(MODELS), len(files)
assert all({f"{m}_vote" for m in P6_METHODS} <= set(np.load(os.path.join(P6, "streams", f)).files) for f in files)
s = build_stream("tilt_4", 2, 3)
r = np.load(stream_path("tilt_4", 2, 3))
for method in P6_METHODS:
    assert np.array_equal(run6(s, 2, 3, method)["vote"], r[f"{method}_vote"]), method
print("225 Phase 6 stream files; re-run reproduces the votes")

  0%|          | 0/225 [00:00<?, ?it/s]

  0%|          | 1/225 [00:01<06:57,  1.86s/it]

  1%|          | 2/225 [00:03<06:44,  1.81s/it]

  1%|▏         | 3/225 [00:05<06:40,  1.80s/it]

  2%|▏         | 4/225 [00:07<06:36,  1.79s/it]

...

 99%|█████████▉| 223/225 [04:32<00:01,  1.09it/s]

100%|█████████▉| 224/225 [04:33<00:00,  1.09it/s]

100%|██████████| 225/225 [04:34<00:00,  1.10it/s]

100%|██████████| 225/225 [04:34<00:00,  1.22s/it]

225 Phase 6 stream files; re-run reproduces the votes


In [7]:
P5_REF = ["source", "continuous", "tent", "triggered"]  # frozen Phase 5 rows, read from the Phase 5 files


def macro_f1(y, p):
    return f1_score(y, p, average="macro", labels=range(len(CLASSES)))


rows, gate = [], []
for target, seed, fold in itertools.product(TARGETS, SEEDS, range(K_FOLDS)):
    r5, r6 = p5(target, seed, fold), np.load(stream_path(target, seed, fold))
    y, cp = r5["y"], int(r5["change_point"])
    for method, r, phase in [(m, r5, 5) for m in P5_REF] + [(m, r6, 6) for m in P6_METHODS]:
        c = json.loads(str(r[f"{method}_cost"]))
        rows.append({"target": target, "seed": seed, "fold": fold, "method": method, "phase": phase,
                     "vote_f1": macro_f1(y[cp:], r[f"{method}_vote"][cp:]),
                     "prefix_vote_f1": macro_f1(y[:cp], r[f"{method}_vote"][:cp]),
                     "fwd_per_window": c["fwd_per_window"], "bwd_per_window": c["bwd_per_window"],
                     "updates": c["events"], "on_fraction": c.get("on_fraction", np.nan)})
        if phase == 6 and method != "balanced":
            on = np.repeat(r[f"{method}_on"], P5_HPS["source"].get("M", 250))[:len(y)].astype(bool)
            first = np.flatnonzero(on[cp:])
            gate.append({"target": target, "seed": seed, "fold": fold, "method": method,
                         "prefix_on": on[:cp].mean(), "target_on": on[cp:].mean(),
                         "detected": len(first) > 0, "delay": first[0] if len(first) else np.nan})
runs = pd.DataFrame(rows)
gate = pd.DataFrame(gate)


def pm(df, cols, by):
    g = df.groupby(by, sort=False)[cols].agg(["mean", "std"])
    return pd.DataFrame({c: g[(c, "mean")].map("{:.4f}".format) + " ± " + g[(c, "std")].map("{:.4f}".format) for c in cols})


def paired(target, a, b, col):
    k = ["seed", "fold"]
    x = runs[(runs.target == target) & (runs.method == a)].set_index(k)[col]
    return paired_test(x, runs[(runs.target == target) & (runs.method == b)].set_index(k)[col].reindex(x.index))


cmp = []
for target, method in itertools.product(TARGETS, P5_REF + P6_METHODS):
    row = {"target": target, "method": method}
    for ref in ("source", "continuous"):
        for col, tag in (("vote_f1", "target"), ("prefix_vote_f1", "prefix")):
            t = paired(target, method, ref, col)
            row.update({f"{tag}_diff_{ref}": t["diff"], f"{tag}_ci_lo_{ref}": t["ci_lo"],
                        f"{tag}_ci_hi_{ref}": t["ci_hi"], f"{tag}_p_{ref}": t["p"]})
    cmp.append(row)
table = (pm(runs, ["vote_f1", "prefix_vote_f1"], ["target", "method"])
         .join(runs.groupby(["target", "method"], sort=False).phase.first())
         .join(pd.DataFrame(cmp).set_index(["target", "method"])))
assert len(table) == len(TARGETS) * len(P5_REF + P6_METHODS)
table.to_csv(os.path.join(P6, "gated_e3.csv"))
table[["phase", "vote_f1", "prefix_vote_f1", "target_diff_source", "prefix_diff_source", "target_diff_continuous",
       "prefix_diff_continuous"]].round(4)

phase          vote_f1   prefix_vote_f1  target_diff_source  prefix_diff_source  target_diff_continuous  prefix_diff_continuous
target method                                                                                                                                           
BLUE   source                5  0.7611 ± 0.0270  0.8469 ± 0.0434              0.0000              0.0000                  0.0725                  0.0935
       continuous            5  0.6886 ± 0.0332  0.7534 ± 0.0691             -0.0725             -0.0935                  0.0000                  0.0000
       tent                  5  0.6891 ± 0.0325  0.7537 ± 0.0702             -0.0719             -0.0932                  0.0005                  0.0004
       triggered             5  0.7525 ± 0.0457  0.8466 ± 0.0434             -0.0085             -0.0003                  0.0640                  0.0933
       balanced              6  0.7702 ± 0.0281  0.8535 ± 0.0357              0.0091              0.0066                  0.0816                  0.1002
...                        ...              ...              ...                 ...                 ...                     ...                     ...
tilt_4 triggered             5  0.4369 ± 0.1026  0.8444 ± 0.0392              0.0236             -0.0004                 -0.1687                  0.1074
       balanced              6  0.4335 ± 0.0686  0.8475 ± 0.0387              0.0202              0.0026                 -0.1721                  0.1105
       gated_continuous      6  0.4399 ± 0.1286  0.8449 ± 0.0392              0.0266              0.0000                 -0.1657                  0.1079
       gated_balanced        6  0.4133 ± 0.0741  0.8449 ± 0.0392              0.0000              0.0000                 -0.1923                  0.1079
       oracle                6  0.6353 ± 0.0998  0.8449 ± 0.0392              0.2219              0.0000                  0.0296                  0.1079

[72 rows x 7 columns]

In [8]:
gate_t = gate.groupby(["target", "method"], sort=False).agg(
    prefix_on=("prefix_on", "mean"), target_on=("target_on", "mean"), detected=("detected", "mean"),
    delay_median=("delay", "median"), delay_mean=("delay", "mean")).reset_index()
assert set(gate_t.target) == set(TARGETS)
gate_t.to_csv(os.path.join(P6, "gate.csv"), index=False)
cost = runs.groupby("method", sort=False).agg(
    phase=("phase", "first"), fwd_per_window=("fwd_per_window", "mean"), bwd_per_window=("bwd_per_window", "mean"),
    updates_per_stream=("updates", "mean"), on_fraction=("on_fraction", "mean"))
cost = cost[cost.phase == 6]
assert len(cost) == len(P6_METHODS)
cost.to_csv(os.path.join(P6, "cost.csv"))
display(cost.round(3))
gate_t.round(3)

,phase,fwd_per_window,bwd_per_window,updates_per_stream,on_fraction
method,,,,,
balanced,6,5.725,0.0,155.222,1.000
gated_continuous,6,1.030,0.0,5.707,0.046
gated_balanced,6,1.000,0.0,0.000,0.046
oracle,6,1.442,0.0,99.658,0.506


,target,method,prefix_on,target_on,detected,delay_median,delay_mean
0,BLUE,gated_continuous,0.000,0.000,0.00,NaN,NaN
1,BLUE,gated_balanced,0.000,0.000,0.00,NaN,NaN
2,BLUE,oracle,0.010,1.000,1.00,0.0,0.000
3,WIFI,gated_continuous,0.000,0.000,0.00,NaN,NaN
4,WIFI,gated_balanced,0.000,0.000,0.00,NaN,NaN
5,WIFI,oracle,0.010,1.000,1.00,0.0,0.000
6,BOTH,gated_continuous,0.000,0.000,0.00,NaN,NaN
7,BOTH,gated_balanced,0.000,0.000,0.00,NaN,NaN
8,BOTH,oracle,0.010,1.000,1.00,0.0,0.000
9,awgn_2,gated_continuous,0.000,0.000,0.00,NaN,NaN
